In [11]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
import matplotlib.pyplot as plt
# seaborn is a better tool for this with more examples 
import seaborn as sns


df_melanoma = pd.read_csv('melanoma_mitoses.csv', sep=';')

In [12]:
df_melanoma.head()

,ID_prog,age,sex,year,histology,site,pT,pN,pM,TNM_stage,breslow,clark,ulcerazione,regressione,til,growth,positive_slnb,mitoses,status,survival
0,1,31,F,2015,Nodular melanoma,Trunk,pT4,N3c,M0,III,>=4,IV,Present,Absent,Absent,Vertical,0,3,0,648
1,2,53,M,2017,Superficial spreading melanoma,Trunk,pT1,N0,M0,I,< 0.76,II,Absent,Present,Present,Vertical,0,0,0,1147
2,3,67,F,2015,Superficial spreading melanoma,Upper limb,pT1,N0,M0,I,0.76-1.50,IV,Absent,Absent,Present,Vertical,0,0,0,1948
3,4,29,F,2015,Superficial spreading melanoma,Trunk,pT1,N1a,M0,III,< 0.76,III,Absent,Absent,Present,Vertical,1,0,0,2170
4,5,57,F,2015,Superficial spreading melanoma,Lower limb,pT1,N0,M0,I,< 0.76,II,Absent,Absent,Present,Radial,0,0,0,1960


In [13]:
df_melanoma['status'].unique()

array([0, 1])

In [14]:
df_melanoma.shape

(2255, 20)

In [26]:
df_melanoma.columns

Index(['ID_prog', 'age', 'sex', 'year', 'histology', 'site', 'pT', 'pN', 'pM',
       'TNM_stage', 'breslow', 'clark', 'ulcerazione', 'regressione', 'til',
       'growth', 'positive_slnb', 'mitoses', 'status', 'survival'],
      dtype='object')

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Prepare data
# X here is the training data which must be a table, so 2D. Here is a good explaination..
#The 1D vs 2D distinction is purely about shape:
#  - A Series [1, 2, 3] has shape (3,) — 1D
#  - A DataFrame with one column has shape (3, 1) — 2D
#  - A DataFrame with two columns has shape (3, 2) — 2D

#  So in your case, Season is not the row identifier — it's a feature just like Operator. The row index is separate and
#  implicit. sklearn doesn't use it at all; it just sees a table of values where each row is one observation and each
#  column is one feature.
#  so we could have X = df_new[['Season', 'Operator', 'Locomotive'.....]] we are just creating a table.

X = df_melanoma[['ID_prog', 'age', 'sex', 'year', 'histology', 'site', 'pT', 'pN', 'pM',
       'TNM_stage', 'breslow', 'clark', 'ulcerazione', 'regressione', 'til',
       'growth', 'positive_slnb', 'mitoses']]

# only if you need to encode your categories
# ski-kitlearn can't cope with catagories such as "spring" so we have to encode them into 
# expand your 2-column X into ~27 binary columns (4 seasons + 24 operators — minus one each to avoid
# redundancy, a concept called the dummy variable trap). get_dummies handles this with drop_first=True:
# X_encoded = pd.get_dummies(X, columns=['Season', 'Operator'], drop_first=True)


# for Y that must be a series (aka a list) which is expected for target variable.
y = df_melanoma['status']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 1804
Test set size: 451


In [16]:
# Fit logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# View learned parameters
print("Learned parameters:")
print(f"  Intercept (β₀): {model.intercept_[0]:.4f}")
print(f"  Coefficient (β₁): {model.coef_[0][0]:.4f}")
print(f"\nDecision boundary: {-model.intercept_[0]/model.coef_[0][0]:.2f}")

ValueError: could not convert string to float: 'F'

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Late', 'Ontime']))